# PhyDG-Slide: full experiment grid on a GPU (Colab or Kaggle)

1. Runtime -> change runtime type -> GPU (T4 or better). On Kaggle: Settings -> Accelerator -> GPU, Internet on.
2. Run the cells in order. Runs that finished are skipped, so after a disconnect just run everything again.
3. Data (~15 GB of NetCDF after filtering, ~3.7 GB cache) needs ~20 GB of disk. Colab gives ~100 GB; on Kaggle use /kaggle/temp or mount persistent storage.

To keep results across sessions on Colab, set `WORK` to a folder in Google Drive (mount it first).

In [ ]:
import os
WORK = "/content" if os.path.exists("/content") else "/kaggle/working"
REPO = f"{WORK}/abhishekjadhavcs-afk-repo"
BRANCH = "claude/phydg-slide"
%cd {WORK}
!test -d {REPO} || git clone -b {BRANCH} https://github.com/abhishekjadhavcs-afk/abhishekjadhavcs-afk-repo {REPO}
%cd {REPO}
!git pull -q
!pip install -q xarray h5netcdf netCDF4
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Stream the S2 archives (~39 GB downloaded, ~15 GB kept). Resumable.
!python scripts/download_s2.py --out data/data_harmonized

In [ ]:
# Precompute composites once (~3.7 GB). Resumable.
!python scripts/build_cache.py --data-root data/data_harmonized --out data/cache --workers 4

In [ ]:
!python -m pytest -q tests

## Stage 1: main comparison (U-Net vs PhyDG-Slide, 8 folds x 3 seeds)
About 30 min per run on a T4 (estimate), so ~24 h for 48 runs. Use fewer seeds first if GPU hours are limited.

In [ ]:
!python scripts/run_grid.py --variants unet phydg --seeds 42 123 777 -- --workers 4 --epochs 60

## Stage 2: ablations (single seed)

In [ ]:
!python scripts/run_grid.py --variants phydg-nostyle phydg-noibn phydg-nofilm phydg-nopoe phydg-novrex --seeds 42 -- --workers 4 --epochs 60

In [ ]:
!python scripts/run_grid.py --summary-only
!tar -czf results.tar.gz runs/*/*/results.json && ls -lh results.tar.gz